# 04 — Monte Carlo Resource Assessment

Probabilistic volumetric heat-in-place estimation with sensitivity analysis.

**Objectives:**
- Set up reservoir parameters with uncertainty ranges
- Run Monte Carlo simulation (10,000 iterations)
- Report P10/P50/P90 estimates for heat and electric capacity
- Identify key uncertainty drivers through sensitivity analysis

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.resource_assessment import VolumetricAssessment

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 4.1 Define Reservoir Parameters

Each parameter is defined as a triangular distribution (min, mode, max) reflecting geological uncertainty.

In [ ]:
# Initialize assessment
assessment = VolumetricAssessment(n_simulations=10000, seed=42)

# Set parameters for a hypothetical BCS geothermal prospect
assessment.set_parameters(
    area_km2=(5.0, 10.0, 20.0),            # Reservoir area
    thickness_m=(500, 1000, 2000),           # Reservoir thickness
    temperature_c=(180, 220, 260),           # Reservoir temperature
    porosity=(0.05, 0.10, 0.20),             # Rock porosity
    recovery_factor=(0.05, 0.15, 0.25),      # Recovery factor
    rock_density_kg_m3=2700.0,               # Typical volcanic rock
    rock_heat_capacity_j_kg_k=900.0,
    reference_temp_c=20.0,                    # Surface temperature (BCS)
    conversion_efficiency=0.12,               # Binary plant efficiency
)

print('Reservoir parameters configured')

## 4.2 Run Monte Carlo Simulation

In [ ]:
# Execute simulation
results = assessment.run_simulation()

# Full report
assessment.print_report()

## 4.3 Distribution of Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = [
    ('heat_in_place_PJ', 'Heat in Place (PJ)', 'steelblue'),
    ('recoverable_heat_PJ', 'Recoverable Heat (PJ)', 'darkorange'),
    ('electric_capacity_MWe', 'Electric Capacity (MWe)', 'forestgreen'),
]

stats = assessment.get_statistics()

for ax, (col, title, color) in zip(axes, metrics):
    ax.hist(results[col], bins=60, color=color, alpha=0.7, edgecolor='white')
    s = stats[col]
    ax.axvline(s['P10'], color='red', linestyle='--', label=f'P10: {s["P10"]:.1f}')
    ax.axvline(s['P50'], color='black', linestyle='-', linewidth=2, label=f'P50: {s["P50"]:.1f}')
    ax.axvline(s['P90'], color='green', linestyle='--', label=f'P90: {s["P90"]:.1f}')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_ylabel('Frequency')

plt.suptitle('Monte Carlo Resource Assessment — 10,000 Simulations', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../images/04_monte_carlo_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.4 Sensitivity Analysis (Tornado Diagram)

In [ ]:
# Sensitivity analysis
sensitivity = assessment.sensitivity_analysis()
print('Sensitivity Analysis (Spearman Rank Correlation):')
print(sensitivity.to_string(index=False))

# Tornado diagram
fig, ax = plt.subplots(figsize=(10, 5))
sensitivity_sorted = sensitivity.sort_values('abs_correlation')

colors = ['#e74c3c' if c > 0 else '#3498db' for c in sensitivity_sorted['spearman_correlation']]
ax.barh(sensitivity_sorted['input_parameter'], sensitivity_sorted['spearman_correlation'], color=colors)
ax.set_xlabel('Spearman Rank Correlation', fontsize=12)
ax.set_title('Sensitivity Analysis — Key Uncertainty Drivers', fontsize=14, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.5)

for i, (_, row) in enumerate(sensitivity_sorted.iterrows()):
    ax.text(row['spearman_correlation'] + 0.01 * np.sign(row['spearman_correlation']),
            i, f'{row["spearman_correlation"]:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('../images/04_sensitivity_tornado.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.5 Parameter Correlation Matrix

In [ ]:
# Pairplot of key parameters vs output
cols = ['area_km2', 'thickness_m', 'temperature_c', 'recovery_factor', 'electric_capacity_MWe']
sample = results[cols].sample(2000, random_state=42)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
inputs = ['area_km2', 'thickness_m', 'temperature_c', 'recovery_factor']

for ax, inp in zip(axes.flat, inputs):
    ax.scatter(sample[inp], sample['electric_capacity_MWe'], alpha=0.2, s=5, color='steelblue')
    ax.set_xlabel(inp.replace('_', ' ').title(), fontsize=11)
    ax.set_ylabel('Electric Capacity (MWe)', fontsize=11)
    # Add trendline
    z = np.polyfit(sample[inp], sample['electric_capacity_MWe'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(sample[inp].min(), sample[inp].max(), 100)
    ax.plot(x_line, p(x_line), 'r-', linewidth=2)

plt.suptitle('Parameter vs Electric Capacity Correlation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/04_parameter_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.6 Key Findings

| Metric | P10 (Conservative) | P50 (Median) | P90 (Optimistic) |
|--------|-------------------|-------------|------------------|
| Heat in Place | See output above | — | — |
| Recoverable Heat | — | — | — |
| Electric Capacity | — | — | — |

**Key uncertainty drivers** (from sensitivity analysis):
1. Reservoir area and thickness contribute most to output variance
2. Temperature is the third most influential parameter
3. Recovery factor introduces significant uncertainty — field-specific calibration is critical

---
**End of notebook series.** For the full toolkit documentation, see [docs/methodology.md](../docs/methodology.md).